# Subject Matter Knowledge (SMK) Evaluator

**The Subject Matter Knowledge (SMK) Evaluator** assesses the background knowledge demands of informational text for students in grades 3–11. When you run a passage through the evaluator, it returns a structured output that includes:

* **complexity_score**: The SMK complexity level (Slightly to Exceedingly Complex).
* **identified_topics**: The core subjects and concepts present in the text.
* **curriculum_check**: Whether topics are standard K-8 content or high school–level specialised knowledge.
* **assumptions_and_scaffolding**: What the author assumes the reader already knows vs. what is explained.
* **friction_analysis**: Whether reading difficulty stems from vocabulary/structure or actual knowledge demands.
* **reasoning**: A brief synthesis of why the text fits the chosen complexity level.

This gives you a clear signal about the knowledge demands of a passage, helping ensure AI-generated content is appropriate for the target grade.

### Install & Load necessary packages

In [1]:
%pip install -qU langchain-google-genai langchain pydantic textstat

Note: you may need to restart the kernel to use updated packages.


In [2]:
import getpass
import os
from typing import List, Literal

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from textstat import textstat as ts

### Set up the evaluator's model and prompts

In [4]:
from prompts import smk_prompts as prompts

# Set your api key in your environment, .env file, or enter when prompted.
# os.environ['GOOGLE_API_KEY'] = 'YOUR API KEY'
load_dotenv()

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

MODEL_NAME = "gemini-3-flash-preview"
TEMPERATURE = 0
model = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=TEMPERATURE)

### Set up the output structure

In [5]:
class SmkOutput(BaseModel):
    identified_topics: List[str] = Field(
        description="List of major subjects/concepts found in the text."
    )
    curriculum_check: str = Field(
        description="Analysis of whether these topics are new or review based on the Grade Level and Reference list."
    )
    assumptions_and_scaffolding: str = Field(
        description="Analysis of what the author assumes vs. what is explained (and if definitions are provided)."
    )
    friction_analysis: str = Field(
        description="Explicit statement distinguishing if difficulty comes from Vocabulary/Structure or actual Knowledge."
    )
    complexity_score: Literal[
        "slightly_complex",
        "moderately_complex",
        "very_complex",
        "exceedingly_complex"
    ] = Field(description="The subject matter knowledge complexity level of the text")
    reasoning: str = Field(
        description="A brief synthesis of why the text fits the chosen complexity level."
    )


prompt_vars = {
    "inputVars": ["text", "grade", "fk_score"],
    "outputParser": JsonOutputParser(pydantic_object=SmkOutput),
}

### Define text complexity evaluation function

In [6]:
def calculate_fk_score(text) -> float:
    """
    Calculate the Flesch-Kincaid Grade Level
    """
    fk_score = round(ts.flesch_kincaid_grade(text), 2)

    return fk_score


def predict_text_complexity_level(text, grade):
    dataset = {
        "text": text,
        "grade": grade,
        "fk_score": calculate_fk_score(text),
    }

    messages = [
        SystemMessage(content=prompts.smk_system_prompt),
        HumanMessagePromptTemplate.from_template(prompts.smk_user_prompt),
    ]

    prompt = ChatPromptTemplate(
        messages,
        input_variables=prompt_vars["inputVars"],
        partial_variables={
            "format_instructions": prompt_vars["outputParser"].get_format_instructions()
        },
    )

    chain = prompt | model | JsonOutputParser()
    return chain.invoke(dataset)

# Test out examples

In [7]:
# Add your text & the grade level you want to evaluate for SMK complexity

sample_text = """
"Well, then," said the teacher, "you may take your slate and go out behind the schoolhouse for half an hour. Think of something to write about, and write the word on your slate. Then try to tell what it is, what it is like, what it is good for, and what is done with it. That is the way to write a composition." Henry took his slate and went out. Just behind the schoolhouse was Mr. Finney's barn. Quite close to the barn was a garden. And in the garden, Henry saw a turnip. "Well, I know what that is," he said to himself; and he wrote the word turnip on his slate. Then he tried to tell what it was like, what it was good for, and what was done with it. Before the half hour was ended he had written a very neat composition on his slate. He then went into the house, and waited while the teacher read it. The teacher was surprised and pleased. He said, "Henry Longfellow, you have done very well. Today you may stand up before the school and read what you have written about the turnip."
"""

result = predict_text_complexity_level(sample_text, 4)
display(result)

{'identified_topics': ['School life',
  'Writing process (composition)',
  'Historical school tools (slate)',
  'Everyday objects (turnip, barn)'],
 'curriculum_check': 'Standard/General. The topics of school assignments and basic writing instructions are foundational to elementary education (K-5).',
 'assumptions_and_scaffolding': "The author assumes the reader understands the basic setting of a schoolhouse and can infer the function of a 'slate' from the context of writing. The text provides a clear scaffold for the writing process (what it is, what it's like, what it's good for), making the concept of a 'composition' very accessible.",
 'friction_analysis': 'There is no friction between concrete and abstract elements. The text remains entirely concrete, describing a physical task (writing on a slate) about a physical object (a turnip) to achieve a concrete result (a finished composition).',
 'complexity_score': 'slightly_complex',
 'reasoning': "The text relies on everyday, practica

You can copy or edit the above cell to test out different texts and grade levels.